![Identidade visual da disciplina](../capa-analise-visualizacao-python-ia.png)

# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Introdução ao Altair

**Objetivo:** construir visualizações declarativas, combinar gráficos e adicionar interatividade com Altair.

**Dados:** conjunto fictício de vendas, criado no próprio notebook para que todos os exemplos sejam reproduzíveis sem acesso à internet.

### Roteiro

1. Estrutura de um gráfico no Altair
2. Barras, linhas, dispersão e distribuições
3. Agregações e transformações
4. Composição de gráficos
5. Interatividade

## 1. Preparação do ambiente

Altair descreve o gráfico em Python e o renderiza com Vega-Lite. Se necessário, descomente a instalação abaixo.

In [1]:
# Se necessário, descomente e execute:
# %pip install altair pandas

import altair as alt
import pandas as pd

print("Altair:", alt.__version__)
print("pandas:", pd.__version__)

Altair: 4.2.2
pandas: 1.3.5


### Dados utilizados

Cada linha representa as vendas de uma categoria em uma região e mês. Também há medidas de investimento em publicidade, unidades vendidas e avaliação dos clientes.

In [2]:
meses = pd.date_range("2025-01-01", periods=12, freq="MS")
categorias = ["Eletrônicos", "Casa", "Livros"]
regioes = ["Sul", "Sudeste"]
linhas = []

for i, mes in enumerate(meses):
    for j, categoria in enumerate(categorias):
        for k, regiao in enumerate(regioes):
            unidades = 72 + 7*i + 18*j + 12*k + ((i + 2*j) % 4)*6
            preco = [320, 140, 55][j]
            linhas.append({
                "mes": mes, "categoria": categoria, "regiao": regiao,
                "unidades": unidades, "receita": unidades * preco,
                "publicidade": 3200 + 310*i + 900*j + 450*k,
                "avaliacao": round(3.55 + 0.06*i + 0.18*j - 0.08*k, 2)
            })

vendas = pd.DataFrame(linhas)
vendas.head()

,mes,categoria,regiao,unidades,receita,publicidade,avaliacao
0,2025-01-01,Eletrônicos,Sul,72,23040,3200,3.55
1,2025-01-01,Eletrônicos,Sudeste,84,26880,3650,3.47
2,2025-01-01,Casa,Sul,102,14280,4100,3.73
3,2025-01-01,Casa,Sudeste,114,15960,4550,3.65
4,2025-01-01,Livros,Sul,108,5940,5000,3.91


In [3]:
vendas

,mes,categoria,regiao,unidades,receita,publicidade,avaliacao
0,2025-01-01,Eletrônicos,Sul,72,23040,3200,3.55
1,2025-01-01,Eletrônicos,Sudeste,84,26880,3650,3.47
2,2025-01-01,Casa,Sul,102,14280,4100,3.73
3,2025-01-01,Casa,Sudeste,114,15960,4550,3.65
4,2025-01-01,Livros,Sul,108,5940,5000,3.91
...,...,...,...,...,...,...,...
67,2025-12-01,Eletrônicos,Sudeste,179,57280,7060,4.13
68,2025-12-01,Casa,Sul,173,24220,7510,4.39
69,2025-12-01,Casa,Sudeste,185,25900,7960,4.31
70,2025-12-01,Livros,Sul,203,11165,8410,4.57


## 2. A gramática do Altair

Um gráfico combina três elementos: **dados**, uma **marca** visual e **codificações** que associam variáveis a canais como posição, cor e tamanho.

Os sufixos indicam o tipo semântico: `N` nominal, `O` ordinal, `Q` quantitativo e `T` temporal.

In [4]:
alt.Chart(vendas).mark_point(size=80).encode(
    x=alt.X("publicidade:Q", title="Investimento em publicidade (R$)"),
    y=alt.Y("receita:Q", title="Receita (R$)"),
    color=alt.Color("categoria:N", title="Categoria"),
    tooltip=["mes:T", "categoria:N", "regiao:N", "receita:Q"]
).properties(
    width=650, height=330, title="Publicidade e receita"
)

alt.Chart(...)

O resultado acima é declarativo: informamos **o que** cada canal representa, e o Altair define escalas, eixos e legenda.

## 3. Comparação — gráfico de barras

Agregações podem ser declaradas diretamente na codificação. Aqui, o comprimento das barras representa a soma da receita por categoria.

In [5]:
alt.Chart(vendas).mark_bar().encode(
    x=alt.X("sum(receita):Q", title="Receita total (R$)"),
    y=alt.Y("categoria:N", title=None, sort="-x"),
    color=alt.Color("categoria:N", legend=None),
    tooltip=["categoria:N", alt.Tooltip("sum(receita):Q", format=",.0f")]
).properties(width=650, height=220, title="Receita total por categoria")

alt.Chart(...)

## 4. Evolução temporal — gráfico de linhas

Linhas são adequadas quando a ordem temporal faz parte da pergunta. A cor separa as categorias.

In [6]:
alt.Chart(vendas).mark_line(point=True).encode(
    x=alt.X("yearmonth(mes):T", title="Mês"),
    y=alt.Y("sum(receita):Q", title="Receita mensal (R$)", scale=alt.Scale(zero=False)),
    color=alt.Color("categoria:N", title="Categoria"),
    tooltip=[alt.Tooltip("yearmonth(mes):T", title="Mês"), "categoria:N", alt.Tooltip("sum(receita):Q", format=",.0f")]
).properties(width=650, height=320, title="Evolução mensal da receita")

alt.Chart(...)

## 5. Relação entre medidas — dispersão

A posição mostra duas medidas; cor e forma acrescentam categorias. O tamanho foi mantido constante para evitar informação redundante.

In [7]:
alt.Chart(vendas).mark_circle(size=90, opacity=0.75).encode(
    x=alt.X("publicidade:Q", title="Publicidade (R$)", scale=alt.Scale(zero=False)),
    y=alt.Y("unidades:Q", title="Unidades vendidas", scale=alt.Scale(zero=False)),
    color=alt.Color("categoria:N", title="Categoria"),
    shape=alt.Shape("regiao:N", title="Região"),
    tooltip=["mes:T", "categoria:N", "regiao:N", "publicidade:Q", "unidades:Q"]
).properties(width=650, height=330, title="Publicidade e unidades vendidas")

alt.Chart(...)

## 6. Distribuições — histograma e boxplot

O histograma agrupa valores em intervalos. O boxplot resume mediana, quartis e valores extremos. São visões complementares.

In [8]:
histograma = alt.Chart(vendas).mark_bar().encode(
    x=alt.X("unidades:Q", bin=alt.Bin(maxbins=15), title="Unidades vendidas"),
    y=alt.Y("count():Q", title="Número de observações"),
    tooltip=[alt.Tooltip("count():Q", title="Observações")]
).properties(width=360, height=260, title="Histograma")

boxplot = alt.Chart(vendas).mark_boxplot(size=35).encode(
    x=alt.X("categoria:N", title=None),
    y=alt.Y("unidades:Q", title="Unidades vendidas"),
    color=alt.Color("categoria:N", legend=None)
).properties(width=360, height=260, title="Boxplot por categoria")

histograma | boxplot

alt.HConcatChart(...)

## 7. Pequenos múltiplos

O facetamento repete a mesma estrutura para cada região, facilitando comparações sem sobrepor todas as séries.

In [9]:
alt.Chart(vendas).mark_line(point=True).encode(
    x=alt.X("yearmonth(mes):T", title="Mês"),
    y=alt.Y("sum(unidades):Q", title="Unidades", scale=alt.Scale(zero=False)),
    color=alt.Color("categoria:N", title="Categoria"),
    column=alt.Column("regiao:N", title="Região")
).properties(width=310, height=260, title="Vendas mensais por região")

alt.Chart(...)

## 8. Camadas e transformação

Camadas permitem sobrepor marcas. O exemplo combina a série mensal com uma linha tracejada para a média de todo o período.

In [10]:
mensal = alt.Chart(vendas).mark_line(point=True, color="#005580").encode(
    x=alt.X("yearmonth(mes):T", title="Mês"),
    y=alt.Y("sum(receita):Q", title="Receita mensal (R$)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("yearmonth(mes):T", title="Mês"), alt.Tooltip("sum(receita):Q", format=",.0f")]
)

media = alt.Chart(vendas).transform_aggregate(
    receita_mensal="sum(receita)", groupby=["mes"]
).transform_joinaggregate(
    media_receita="mean(receita_mensal)"
).mark_rule(color="#D55E00", strokeDash=[6, 4]).encode(
    y=alt.Y("media_receita:Q"),
    tooltip=[alt.Tooltip("media_receita:Q", title="Média mensal", format=",.0f")]
)

(mensal + media).properties(width=650, height=320, title="Receita mensal e média do período")

alt.LayerChart(...)

## 9. Interatividade com seleção

Arraste horizontalmente para selecionar um intervalo de meses. Os pontos dentro da seleção permanecem destacados; os demais ficam esmaecidos. Clique fora da área selecionada para limpar a seleção.

In [11]:
intervalo = alt.selection_interval(encodings=["x"], name="intervalo_meses")

grafico_selecao = alt.Chart(vendas).mark_circle(size=105).encode(
    x=alt.X("yearmonth(mes):T", title="Mês"),
    y=alt.Y("receita:Q", title="Receita (R$)", scale=alt.Scale(zero=False)),
    color=alt.Color("categoria:N", title="Categoria"),
    shape=alt.Shape("regiao:N", title="Região"),
    opacity=alt.condition(intervalo, alt.value(0.9), alt.value(0.12)),
    tooltip=[
        alt.Tooltip("yearmonth(mes):T", title="Mês"),
        alt.Tooltip("categoria:N", title="Categoria"),
        alt.Tooltip("regiao:N", title="Região"),
        alt.Tooltip("receita:Q", title="Receita", format=",.0f"),
        alt.Tooltip("unidades:Q", title="Unidades")
    ]
).properties(
    width=700, height=350, title="Seleção de um intervalo de meses"
)

# Altair 5+ usa parâmetros; Altair 4 usa seleções.
if hasattr(grafico_selecao, "add_params"):
    grafico_selecao = grafico_selecao.add_params(intervalo)
else:
    grafico_selecao = grafico_selecao.add_selection(intervalo)

grafico_selecao

alt.Chart(...)

## 10. Um tema reutilizável

Configurações comuns podem ser aplicadas ao gráfico final. Isso ajuda a manter tipografia, cores e títulos consistentes.

In [12]:
grafico_final = alt.Chart(vendas).mark_bar(cornerRadiusEnd=3).encode(
    x=alt.X("sum(unidades):Q", title="Unidades vendidas"),
    y=alt.Y("regiao:N", title=None),
    color=alt.Color("categoria:N", title="Categoria"),
    order=alt.Order("categoria:N"),
    tooltip=["regiao:N", "categoria:N", alt.Tooltip("sum(unidades):Q", title="Unidades")]
).properties(width=650, height=220, title="Composição das vendas por região").configure(
    background="#FAFAFA"
).configure_axis(
    labelFontSize=12, titleFontSize=13, gridColor="#E5E5E5"
).configure_title(
    fontSize=18, anchor="start", color="#17324D"
)

grafico_final

alt.Chart(...)

## Síntese

- Escolha a marca de acordo com a pergunta: comparação, evolução, relação ou distribuição.
- Declare corretamente os tipos `N`, `O`, `Q` e `T`.
- Use agregações e transformações sem alterar a tabela original.
- Combine gráficos com `+`, `|` e `&`.
- Acrescente interação apenas quando ela apoiar uma tarefa clara.

**Para experimentar:** troque uma codificação, aplique um filtro ou crie outra visão para responder a uma pergunta sobre os dados.

### Referências

- [Documentação do Altair](https://altair-viz.github.io/)
- [Galeria de exemplos](https://altair-viz.github.io/gallery/index.html)
- [Documentação do Vega-Lite](https://vega.github.io/vega-lite/)